In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
np.random.seed(42)


In [ ]:
import os
# The lecture file data_1.csv (CO2, temperature and humidity every minute) is not part of the repository.
# If it is missing, a synthetic file with the same columns is written so that the notebook runs end to end.
if not os.path.exists("data_1.csv"):
    rng = np.random.default_rng(0)
    t = np.arange(1008)
    noise = np.convolve(rng.normal(0, 0.25, 1028), np.ones(10) / 10, mode="same")[10:-10]
    humidity = 40 - 0.004 * t + 5 * np.sin(2 * np.pi * t / 480) + 1.5 * np.sin(2 * np.pi * t / 97) + noise
    stamp = pd.date_range("2023-11-03 18:45", periods=1008, freq="min").strftime("%m/%d/%y %H:%M")
    pd.DataFrame({"Date Time": stamp,
                  "CO2 (ppm)": (900 + 60 * np.sin(t / 90) + rng.normal(0, 5, 1008)).round().astype(int),
                  "Temperature (C)": (28 - 0.005 * t).round(2), "Humidity (%)": humidity.round(2)}
                 ).to_csv("data_1.csv", index=False)


In [ ]:
data = pd.read_csv('data_1.csv')
data.head()

In [ ]:
data.value_counts()

In [ ]:
plt.plot(data["CO2 (ppm)"])
plt.xlabel("Time")
plt.ylabel("CO2 (ppm)")
plt.show()

In [ ]:
plt.plot(data["Temperature (C)"])
plt.xlabel("Time")
plt.ylabel("Temperature (C)")
plt.show()

In [ ]:
plt.plot(data["Humidity (%)"])
plt.xlabel("Time")
plt.ylabel("Humidity (%)")
plt.show()

In [ ]:
### Normalization ###
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
new_data = data.drop(columns=["Date Time"])
data_scaled = scaler.fit_transform(new_data)
data_scaled = pd.DataFrame(data_scaled, columns=new_data.columns)
data_scaled.head()

In [ ]:
X = data_scaled["Humidity (%)"].values
X.shape

In [ ]:
## without normalization ##
# X = data["Humidity (%)"].values
# X.shape

In [ ]:
plt.plot(X)
plt.xlabel("Time")
plt.ylabel("Humidity (%) Normilazed")
plt.show()

In [ ]:
### Prepare the data for RNN ###
def create_dataset(X, time_step=3):
    Xs = []
    ys = []
    for i in range(len(X) - time_step - 1):
        Xs.append(X[i:(i + time_step)])
        ys.append(X[i + time_step])
    return np.array(Xs), np.array(ys)


In [ ]:
X_data, y_data = create_dataset(X, time_step=3)
X_data.shape, y_data.shape

In [ ]:
### split tha data into train and test ###
train_size = int(len(X_data) * 0.8)
test_size = len(X_data) - train_size
X_train, X_test = X_data[0:train_size], X_data[train_size:len(X_data)]
y_train, y_test = y_data[0:train_size], y_data[train_size:len(y_data)]
X_train.shape, X_test.shape, y_train.shape, y_test.shape

## **One sample implementation**

In [ ]:
sample_x = X_train[0]
sample_y = y_train[0]
print(sample_x, "->", sample_y)

In [ ]:
import torch 
x_tensor = torch.tensor(sample_x).float()
y_tensor = torch.tensor(sample_y).float()

x_tensor, y_tensor

In [ ]:
def pick_device():
    """cuda, then mps, then cpu; a device is used only if a tensor can really be created on it."""
    for name, ok in (("cuda", torch.cuda.is_available), ("mps", torch.backends.mps.is_available)):
        try:
            if ok():
                torch.zeros(1, device=name)
                return torch.device(name)
        except Exception:
            pass
    return torch.device("cpu")

device = pick_device()
print("Using device:", device)

In [ ]:
### Initialize params of LSTM cell ###
g = torch.Generator().manual_seed(42)        # the original torch.Generator(42) is NOT a seed (42 = device index)
def rand_param():
    """uniform [0, 1) parameter drawn from the seeded generator, placed on `device`"""
    return torch.rand(1, generator=g).to(device).requires_grad_()
W_fx = rand_param()
W_fh = rand_param()
b_f = torch.zeros(1, requires_grad=True, device=device)
W_ix = rand_param()
W_ih = rand_param()
b_i = torch.zeros(1, requires_grad=True, device=device)
W_gx = rand_param()
W_gh = rand_param()
b_g = torch.zeros(1, requires_grad=True, device=device)
W_ox = rand_param()
W_oh = rand_param()
b_o = torch.zeros(1, requires_grad=True, device=device)
W_y = rand_param()
b_y = torch.zeros(1, requires_grad=True, device=device)

In [ ]:
### feed forward ###
h_0 = torch.zeros(1, device=device)
c_0 = torch.zeros(1, device=device)
x_tensor = x_tensor.to(device)
y_tensor = y_tensor.to(device)

### t = 1 ###
### forget gate ###
z_f1 = W_fx * x_tensor[0] + W_fh * h_0 + b_f
f_1 = torch.sigmoid(z_f1)
### inputa gate ###
z_i1 = W_ix * x_tensor[0] + W_ih * h_0 + b_i
i_1 = torch.sigmoid(z_i1)
### candidate gate ###
z_g1 = W_gx * x_tensor[0] + W_gh * h_0 + b_g
g_1 = torch.tanh(z_g1)
### output gate ###
z_o1 = W_ox * x_tensor[0] + W_oh * h_0 + b_o
o_1 = torch.sigmoid(z_o1)
### Cell state ###
c_1 = f_1 * c_0 + i_1 * g_1
### Hidden state ###
h_1 = o_1 * torch.tanh(c_1)

### t = 2 ###
### forget gate ###
z_f2 = W_fx * x_tensor[1] + W_fh * h_1 + b_f
f_2 = torch.sigmoid(z_f2)
### inputa gate ###
z_i2 = W_ix * x_tensor[1] + W_ih * h_1 + b_i
i_2 = torch.sigmoid(z_i2)
### candidate gate ###
z_g2 = W_gx * x_tensor[1] + W_gh * h_1 + b_g
g_2 = torch.tanh(z_g2)
### output gate ###
z_o2 = W_ox * x_tensor[1] + W_oh * h_1 + b_o
o_2 = torch.sigmoid(z_o2)
### Cell state ###
c_2 = f_2 * c_1 + i_2 * g_2
### Hidden state ###
h_2 = o_2 * torch.tanh(c_2)

### t = 3 ###
### forget gate ###
z_f3 = W_fx * x_tensor[2] + W_fh * h_2 + b_f
f_3 = torch.sigmoid(z_f3)
### inputa gate ###
z_i3 = W_ix * x_tensor[2] + W_ih * h_2 + b_i
i_3 = torch.sigmoid(z_i3)
### candidate gate ###
z_g3 = W_gx * x_tensor[2] + W_gh * h_2 + b_g
g_3 = torch.tanh(z_g3)
### output gate ###
z_o3 = W_ox * x_tensor[2] + W_oh * h_2 + b_o
o_3 = torch.sigmoid(z_o3)
### Cell state ###
c_3 = f_3 * c_2 + i_3 * g_3
### Hidden state ###
h_3 = o_3 * torch.tanh(c_3)

y_pred = W_y * h_3 + b_y
print("Predicted:", y_pred, "Actual:", y_tensor)

In [ ]:
### Calculating Loss ###
### MSE ###
loss = 0.5 * ((y_pred - y_tensor) ** 2)
print("Loss:", loss*10000)

In [ ]:
### Backpropogation ###
# gradients
for p_ in (W_fx, W_fh, b_f, W_ix, W_ih, b_i, W_gx, W_gh, b_g, W_ox, W_oh, b_o, W_y, b_y):
    p_.grad = None      # otherwise backward() ADDS to the gradient of the previous step
loss.backward()
dL_d_wy = W_y.grad
dL_d_by = b_y.grad
dL_d_W_fx = W_fx.grad
dL_d_W_fh = W_fh.grad
dL_d_bf = b_f.grad
dL_d_W_ix = W_ix.grad
dL_d_W_ih = W_ih.grad
dL_d_bi = b_i.grad
dL_d_W_gx = W_gx.grad
dL_d_W_gh = W_gh.grad
dL_d_bg = b_g.grad
dL_d_W_ox = W_ox.grad
dL_d_W_oh = W_oh.grad
dL_d_bo = b_o.grad 


In [ ]:
### Upfdate parameters ###
lr = 0.1
with torch.no_grad():
    W_y -= lr * dL_d_wy
    b_y -= lr * dL_d_by
    W_fx -= lr * dL_d_W_fx
    W_fh -= lr * dL_d_W_fh
    b_f -= lr * dL_d_bf
    W_ix -= lr * dL_d_W_ix
    W_ih -= lr * dL_d_W_ih
    b_i -= lr * dL_d_bi
    W_gx -= lr * dL_d_W_gx
    W_gh -= lr * dL_d_W_gh
    b_g -= lr * dL_d_bg
    W_ox -= lr * dL_d_W_ox
    W_oh -= lr * dL_d_W_oh
    b_o -= lr * dL_d_bo

In [ ]:
### feed forward ###
h_0 = torch.zeros(1, device=device)
c_0 = torch.zeros(1, device=device)

### t = 1 ###
### forget gate ###
z_f1 = W_fx * x_tensor[0] + W_fh * h_0 + b_f
f_1 = torch.sigmoid(z_f1)
### inputa gate ###
z_i1 = W_ix * x_tensor[0] + W_ih * h_0 + b_i
i_1 = torch.sigmoid(z_i1)
### candidate gate ###
z_g1 = W_gx * x_tensor[0] + W_gh * h_0 + b_g
g_1 = torch.tanh(z_g1)
### output gate ###
z_o1 = W_ox * x_tensor[0] + W_oh * h_0 + b_o
o_1 = torch.sigmoid(z_o1)
### Cell state ###
c_1 = f_1 * c_0 + i_1 * g_1
### Hidden state ###
h_1 = o_1 * torch.tanh(c_1)

### t = 2 ###
### forget gate ###
z_f2 = W_fx * x_tensor[1] + W_fh * h_1 + b_f
f_2 = torch.sigmoid(z_f2)
### inputa gate ###
z_i2 = W_ix * x_tensor[1] + W_ih * h_1 + b_i
i_2 = torch.sigmoid(z_i2)
### candidate gate ###
z_g2 = W_gx * x_tensor[1] + W_gh * h_1 + b_g
g_2 = torch.tanh(z_g2)
### output gate ###
z_o2 = W_ox * x_tensor[1] + W_oh * h_1 + b_o
o_2 = torch.sigmoid(z_o2)
### Cell state ###
c_2 = f_2 * c_1 + i_2 * g_2
### Hidden state ###
h_2 = o_2 * torch.tanh(c_2)

### t = 3 ###
### forget gate ###
z_f3 = W_fx * x_tensor[2] + W_fh * h_2 + b_f
f_3 = torch.sigmoid(z_f3)
### inputa gate ###
z_i3 = W_ix * x_tensor[2] + W_ih * h_2 + b_i
i_3 = torch.sigmoid(z_i3)
### candidate gate ###
z_g3 = W_gx * x_tensor[2] + W_gh * h_2 + b_g
g_3 = torch.tanh(z_g3)
### output gate ###
z_o3 = W_ox * x_tensor[2] + W_oh * h_2 + b_o
o_3 = torch.sigmoid(z_o3)
### Cell state ###
c_3 = f_3 * c_2 + i_3 * g_3
### Hidden state ###
h_3 = o_3 * torch.tanh(c_3)

y_pred = W_y * h_3 + b_y
print("Predicted:", y_pred, "Actual:", y_tensor)

In [ ]:
### Calculating Loss ###
### MSE ###
loss = 0.5 * ((y_pred - y_tensor) ** 2)
print("Loss:", loss*10000)

In [ ]:
### Backpropogation ###
# gradients
for p_ in (W_fx, W_fh, b_f, W_ix, W_ih, b_i, W_gx, W_gh, b_g, W_ox, W_oh, b_o, W_y, b_y):
    p_.grad = None      # otherwise backward() ADDS to the gradient of the previous step
loss.backward()
dL_d_wy = W_y.grad
dL_d_by = b_y.grad
dL_d_W_fx = W_fx.grad
dL_d_W_fh = W_fh.grad
dL_d_bf = b_f.grad
dL_d_W_ix = W_ix.grad
dL_d_W_ih = W_ih.grad
dL_d_bi = b_i.grad
dL_d_W_gx = W_gx.grad
dL_d_W_gh = W_gh.grad
dL_d_bg = b_g.grad
dL_d_W_ox = W_ox.grad
dL_d_W_oh = W_oh.grad
dL_d_bo = b_o.grad 


In [ ]:
### Upfdate parameters ###
lr = 0.1
with torch.no_grad():
    W_y -= lr * dL_d_wy
    b_y -= lr * dL_d_by
    W_fx -= lr * dL_d_W_fx
    W_fh -= lr * dL_d_W_fh
    b_f -= lr * dL_d_bf
    W_ix -= lr * dL_d_W_ix
    W_ih -= lr * dL_d_W_ih
    b_i -= lr * dL_d_bi
    W_gx -= lr * dL_d_W_gx
    W_gh -= lr * dL_d_W_gh
    b_g -= lr * dL_d_bg
    W_ox -= lr * dL_d_W_ox
    W_oh -= lr * dL_d_W_oh
    b_o -= lr * dL_d_bo

In [ ]:
### feed forward ###
h_0 = torch.zeros(1, device=device)
c_0 = torch.zeros(1, device=device)

### t = 1 ###
### forget gate ###
z_f1 = W_fx * x_tensor[0] + W_fh * h_0 + b_f
f_1 = torch.sigmoid(z_f1)
### inputa gate ###
z_i1 = W_ix * x_tensor[0] + W_ih * h_0 + b_i
i_1 = torch.sigmoid(z_i1)
### candidate gate ###
z_g1 = W_gx * x_tensor[0] + W_gh * h_0 + b_g
g_1 = torch.tanh(z_g1)
### output gate ###
z_o1 = W_ox * x_tensor[0] + W_oh * h_0 + b_o
o_1 = torch.sigmoid(z_o1)
### Cell state ###
c_1 = f_1 * c_0 + i_1 * g_1
### Hidden state ###
h_1 = o_1 * torch.tanh(c_1)

### t = 2 ###
### forget gate ###
z_f2 = W_fx * x_tensor[1] + W_fh * h_1 + b_f
f_2 = torch.sigmoid(z_f2)
### inputa gate ###
z_i2 = W_ix * x_tensor[1] + W_ih * h_1 + b_i
i_2 = torch.sigmoid(z_i2)
### candidate gate ###
z_g2 = W_gx * x_tensor[1] + W_gh * h_1 + b_g
g_2 = torch.tanh(z_g2)
### output gate ###
z_o2 = W_ox * x_tensor[1] + W_oh * h_1 + b_o
o_2 = torch.sigmoid(z_o2)
### Cell state ###
c_2 = f_2 * c_1 + i_2 * g_2
### Hidden state ###
h_2 = o_2 * torch.tanh(c_2)

### t = 3 ###
### forget gate ###
z_f3 = W_fx * x_tensor[2] + W_fh * h_2 + b_f
f_3 = torch.sigmoid(z_f3)
### inputa gate ###
z_i3 = W_ix * x_tensor[2] + W_ih * h_2 + b_i
i_3 = torch.sigmoid(z_i3)
### candidate gate ###
z_g3 = W_gx * x_tensor[2] + W_gh * h_2 + b_g
g_3 = torch.tanh(z_g3)
### output gate ###
z_o3 = W_ox * x_tensor[2] + W_oh * h_2 + b_o
o_3 = torch.sigmoid(z_o3)
### Cell state ###
c_3 = f_3 * c_2 + i_3 * g_3
### Hidden state ###
h_3 = o_3 * torch.tanh(c_3)

y_pred = W_y * h_3 + b_y
print("Predicted:", y_pred, "Actual:", y_tensor)

In [ ]:
### Calculating Loss ###
### MSE ###
loss = 0.5 * ((y_pred - y_tensor) ** 2)
print("Loss:", loss*10000)

## **Putting all together**

In [ ]:
import torch
X_train_tensor = torch.tensor(X_train).float().to(device)
y_train_tensor = torch.tensor(y_train).float().to(device)
X_test_tensor = torch.tensor(X_test).float().to(device)
y_test_tensor = torch.tensor(y_test).float().to(device)

In [ ]:
X_train_tensor[0]

In [ ]:
### Initialize params of LSTM cell ###
g = torch.Generator().manual_seed(42)        # the original torch.Generator(42) is NOT a seed (42 = device index)
def rand_param():
    """uniform [0, 1) parameter drawn from the seeded generator, placed on `device`"""
    return torch.rand(1, generator=g).to(device).requires_grad_()
W_fx = rand_param()
W_fh = rand_param()
b_f = torch.zeros(1, requires_grad=True, device=device)
W_ix = rand_param()
W_ih = rand_param()
b_i = torch.zeros(1, requires_grad=True, device=device)
W_gx = rand_param()
W_gh = rand_param()
b_g = torch.zeros(1, requires_grad=True, device=device)
W_ox = rand_param()
W_oh = rand_param()
b_o = torch.zeros(1, requires_grad=True, device=device)
W_y = rand_param()
b_y = torch.zeros(1, requires_grad=True, device=device)

In [ ]:
### Initialize params of LSTM cell ###

epoches = 100
losses = []
lr = 0.001

params = [W_fx, W_fh, b_f,
          W_ix, W_ih, b_i,
          W_gx, W_gh, b_g,
          W_ox, W_oh, b_o,
          W_y, b_y]

for epoch in range(epoches):
    total_loss = 0.0  # Python float accumulator

    # (optional) reinitialize h_0/c_0 here or inside loop per sample
    # h_0 = torch.zeros(1, device=device)
    # c_0 = torch.zeros(1, device=device)

    for i in range(len(X_train_tensor)):
        x_tensor = X_train_tensor[i]
        sample_y = y_train_tensor[i]

        # reset per-sample initial states
        h_0 = torch.zeros(1, device=device)
        c_0 = torch.zeros(1, device=device)

        # ---------------------------
        # Clear previous gradients
        # ---------------------------
        for p in params:
            p.grad = None

        # Feed forward
        ### t = 1 ###
        z_f1 = W_fx * x_tensor[0] + W_fh * h_0 + b_f
        f_1 = torch.sigmoid(z_f1)
        z_i1 = W_ix * x_tensor[0] + W_ih * h_0 + b_i
        i_1 = torch.sigmoid(z_i1)
        z_g1 = W_gx * x_tensor[0] + W_gh * h_0 + b_g
        g_1 = torch.tanh(z_g1)
        z_o1 = W_ox * x_tensor[0] + W_oh * h_0 + b_o
        o_1 = torch.sigmoid(z_o1)
        c_1 = f_1 * c_0 + i_1 * g_1
        h_1 = o_1 * torch.tanh(c_1)

        ### t = 2 ###
        z_f2 = W_fx * x_tensor[1] + W_fh * h_1 + b_f
        f_2 = torch.sigmoid(z_f2)
        z_i2 = W_ix * x_tensor[1] + W_ih * h_1 + b_i
        i_2 = torch.sigmoid(z_i2)
        z_g2 = W_gx * x_tensor[1] + W_gh * h_1 + b_g
        g_2 = torch.tanh(z_g2)
        z_o2 = W_ox * x_tensor[1] + W_oh * h_1 + b_o
        o_2 = torch.sigmoid(z_o2)
        c_2 = f_2 * c_1 + i_2 * g_2
        h_2 = o_2 * torch.tanh(c_2)

        ### t = 3 ###
        z_f3 = W_fx * x_tensor[2] + W_fh * h_2 + b_f
        f_3 = torch.sigmoid(z_f3)
        z_i3 = W_ix * x_tensor[2] + W_ih * h_2 + b_i
        i_3 = torch.sigmoid(z_i3)
        z_g3 = W_gx * x_tensor[2] + W_gh * h_2 + b_g
        g_3 = torch.tanh(z_g3)
        z_o3 = W_ox * x_tensor[2] + W_oh * h_2 + b_o
        o_3 = torch.sigmoid(z_o3)
        c_3 = f_3 * c_2 + i_3 * g_3
        h_3 = o_3 * torch.tanh(c_3)

        y_pred = W_y * h_3 + b_y

        # Calculate loss (scalar tensor)
        loss = 0.5 * ((y_pred - sample_y) ** 2)

        # Accumulate numeric loss
        total_loss += loss.item()

        # Backpropagation
        loss.backward()

        # Update parameters safely (no_grad + .data)
        with torch.no_grad():
            for p in params:
                if p.grad is not None:
                    p.data -= lr * p.grad

    avg_loss = total_loss / len(X_train_tensor)
    losses.append(avg_loss)
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {avg_loss}")


In [ ]:
loss = torch.tensor(losses).cpu()
plt.plot(loss)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.show()


In [ ]:
predictions = []
h_0 = torch.zeros(1, device=device)      # explicit initial state
c_0 = torch.zeros(1, device=device)
for i in range(len(X_test_tensor)):
    x_tensor = X_test_tensor[i]

    # Feed forward
    ### t = 1 ###
    z_f1 = W_fx * x_tensor[0] + W_fh * h_0 + b_f
    f_1 = torch.sigmoid(z_f1)
    z_i1 = W_ix * x_tensor[0] + W_ih * h_0 + b_i
    i_1 = torch.sigmoid(z_i1)
    z_g1 = W_gx * x_tensor[0] + W_gh * h_0 + b_g
    g_1 = torch.tanh(z_g1)
    z_o1 = W_ox * x_tensor[0] + W_oh * h_0 + b_o
    o_1 = torch.sigmoid(z_o1)
    c_1 = f_1 * c_0 + i_1 * g_1
    h_1 = o_1 * torch.tanh(c_1)

    ### t = 2 ###
    z_f2 = W_fx * x_tensor[1] + W_fh * h_1 + b_f
    f_2 = torch.sigmoid(z_f2)
    z_i2 = W_ix * x_tensor[1] + W_ih * h_1 + b_i
    i_2 = torch.sigmoid(z_i2)
    z_g2 = W_gx * x_tensor[1] + W_gh * h_1 + b_g
    g_2 = torch.tanh(z_g2)
    z_o2 = W_ox * x_tensor[1] + W_oh * h_1 + b_o
    o_2 = torch.sigmoid(z_o2)
    c_2 = f_2 * c_1 + i_2 * g_2
    h_2 = o_2 * torch.tanh(c_2)

    ### t = 3 ###
    z_f3 = W_fx * x_tensor[2] + W_fh * h_2 + b_f
    f_3 = torch.sigmoid(z_f3)
    z_i3 = W_ix * x_tensor[2] + W_ih * h_2 + b_i
    i_3 = torch.sigmoid(z_i3)
    z_g3 = W_gx * x_tensor[2] + W_gh * h_2 + b_g
    g_3 = torch.tanh(z_g3)
    z_o3 = W_ox * x_tensor[2] + W_oh * h_2 + b_o
    o_3 = torch.sigmoid(z_o3)
    c_3 = f_3 * c_2 + i_3 * g_3
    h_3 = o_3 * torch.tanh(c_3)

    y_pred = W_y * h_3 + b_y

    predictions.append(y_pred.item())
predictions = np.array(predictions)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(y_test, label="True Values")
plt.plot(predictions, label="Predicted Values", linestyle='--')
plt.xlabel("Samples")
plt.ylabel("Target Value")
plt.title("True vs Predicted (LSTM Regression)")
plt.legend()
plt.grid(True)
plt.show()


## **LSTM with PyTorch**

In [ ]:
X_train_tensor = torch.tensor(X_train).float().to(device)
y_train_tensor = torch.tensor(y_train).float().to(device)
X_test_tensor = torch.tensor(X_test).float().to(device)
y_test_tensor = torch.tensor(y_test).float().to(device)

In [ ]:
X_train_tensor.shape

In [ ]:
y_train_tensor.shape

In [ ]:
X_train_tensor = X_train_tensor.reshape(-1, 3, 1)
X_test_tensor = X_test_tensor.reshape(-1, 3, 1)
# X_train_tensor = X_train_tensor.unsqueeze(-1)
X_train_tensor.shape

In [ ]:
### Building RNN and LSTM with PyTorch ###
import torch
import torch.nn as nn
import torch.optim as optim

class LSTMModel(nn.Module):
    def __init__(self, hidden=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden, num_layers=1, batch_first=True)  # batch_first!
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):                       # x: (samples, time steps, features)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])           # last time step

class RNNModel(nn.Module):
    def __init__(self, hidden=1):
        super().__init__()
        self.rnn = nn.RNN(input_size=1, hidden_size=hidden, num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

torch.manual_seed(42)
model = LSTMModel(hidden=1).to(device)


In [ ]:
def param_count(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print("Trainable parameters:", param_count(model))

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)   # lr 0.01 is too small: the loss hardly moves in 1000 epochs

epochs = 1000
losses = []

model.train()
for epoch in range(epochs):
    optimizer.zero_grad()
    
    y_pred = model(X_train_tensor).squeeze()   
    loss = criterion(y_pred, y_train_tensor)
    
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


In [ ]:
model.eval()
with torch.no_grad():
    y_test_pred = model(X_test_tensor).squeeze().cpu().numpy()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(y_test, label="True")
plt.plot(y_test_pred, label="Predicted")
plt.legend()
plt.xlabel("Sample")
plt.ylabel("Value")
plt.title("LSTM Prediction vs True Values")
plt.show()


## **Multi-Cell RNN and LSTM**

In [ ]:
X_train_tensor = torch.tensor(X_train).float().to(device)
y_train_tensor = torch.tensor(y_train).float().to(device)
X_test_tensor = torch.tensor(X_test).float().to(device)
y_test_tensor = torch.tensor(y_test).float().to(device)

X_train_tensor = X_train_tensor.reshape(-1, X_train_tensor.shape[1], 1)
X_test_tensor = X_test_tensor.reshape(-1, X_train_tensor.shape[1], 1)
# X_train_tensor = X_train_tensor.unsqueeze(-1)
X_train_tensor.shape

In [ ]:
### Multi-cell models: hidden size 5 ###
torch.manual_seed(42)
model = LSTMModel(hidden=5).to(device)        # classes LSTMModel and RNNModel are defined above
# model = RNNModel(hidden=5).to(device)       # uncomment to compare with a vanilla RNN


In [ ]:
def param_count(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print("Trainable parameters:", param_count(model))

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

epochs = 1000
losses = []

model.train()
for epoch in range(epochs):
    optimizer.zero_grad()
    
    y_pred = model(X_train_tensor).squeeze()   
    loss = criterion(y_pred, y_train_tensor)
    
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


In [ ]:
model.eval()
with torch.no_grad():
    y_test_pred = model(X_test_tensor).squeeze().cpu().numpy()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(y_test, label="True")
plt.plot(y_test_pred, label="Predicted")
plt.legend()
plt.xlabel("Sample")
plt.ylabel("Value")
plt.title("LSTM Prediction vs True Values")
plt.show()
